In [1]:
import cv2

In [2]:
import torch

In [3]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\adilz\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
import os

In [19]:
def process_video(video_path, output_video, output_txt):
    # Load YOLOv8 model
    model = YOLO("yolov8n.pt")  # Nano version for speed

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    txt_file = open(output_txt, "w")
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLOv8 detection with only 'person' class (class 0)
        results = model.predict(source=frame, classes=[0], conf=0.25, verbose=False)

        for result in results:
            boxes = result.boxes
            for box in boxes:
                x_min, y_min, x_max, y_max = map(int, box.xyxy[0])
                confidence = float(box.conf[0])

                # Draw bounding box
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

                # Write detection info
                txt_file.write(f"{frame_idx},{x_min},{y_min},{x_max},{y_max}\n")

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    txt_file.close()
    print(f"Processing complete. Output saved to {output_video} and {output_txt}")

# Define paths
video_path = "video2.avi"  # Input video file
output_video = "output2.avi"  # Output video with bounding boxes
output_txt = "bounding_boxes2.txt"  # Output text file with coordinates

# Run the function
process_video(video_path, output_video, output_txt)

Processing complete. Output saved to output2.avi and bounding_boxes2.txt
